# AquaTox-multi prediction

This tutorial predicts new molecules from the released fold checkpoints. It requires the task-specific duration scalers produced during training preprocessing. The notebook only loads those files; it never fits a scaler on query molecules. Predictions are `log10(mg/L)`.

`Duration_Value` uses the dataset's recorded duration units. `effect` must be one of DVP, GRO, ITX, MOR, MPH, POP, REP; `media_type` must be FW, SW, or OTHER.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('src').resolve()))
import pandas as pd
from rfm.predict import TASKS, find_duration_scalers, load_fold_model, predict_ensemble


In [ ]:
# Query rows: no observed labels are needed for new-molecule prediction.
query = pd.DataFrame([{
    'smiles': 'CCO',
    'Duration_Value': 48.0,
    'effect': 'MOR',
    'media_type': 'FW',
}])
query


In [ ]:
# Each fold uses scalers made from that fold's training slice.
# Set PREPROCESSED_DIR to the existing training preprocessing directory.
PREPROCESSED_DIR = Path('preprocessed_graphs_local/aquatox')
folds = []
for fold_id in range(5):
    checkpoint = Path(f'models/aquatox_multi/fold_{fold_id}/best_model.pt')
    scalers = find_duration_scalers(PREPROCESSED_DIR, fold_id)
    folds.append(load_fold_model(checkpoint, scalers))


In [ ]:
predictions = predict_ensemble(folds, query)
predictions


The output includes one `pred_<task>_log10_mg_per_l` column per task and, for an ensemble, the fold-to-fold standard deviation. The released `src/rfm/evaluate.py` command evaluates labelled outer-test rows using the manifest; it is a separate workflow and is not used above. `Duration_Value` is supplied in hours, as in the source dataset.
